
# SQL Warehousing with Databricks

Databricks SQL provides access to a complete set of features, making SQL workload easy and fully compatible with your existing legacy DW!

In this notebook, you'll find some example with more advanced SQL features, such as loops and stored procedures!

In [0]:
%run ./_resources/00-setup

#### Setup for PK/FK demo notebook

Hide this notebook result.

## Configuration file

Please change your catalog and schema here to run the demo on a different catalog.

<!-- Collect usage data (view). Remove it to disable collection or disable tracker during installation. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=DBSQL&org_id=2162748966026566&notebook=%2F01-Advanced-SQL-Warehouse&demo_name=sql-warehouse&event=VIEW&path=%2F_dbdemos%2FDBSQL%2Fsql-warehouse%2F01-Advanced-SQL-Warehouse&version=1">


# Technical Setup notebook. Hide this cell results
Initialize dataset to the current user and cleanup data when reset_all_data is set to true

Do not edit

USE CATALOG `main__build`
using catalog.database `main__build`.`dbdemos_pk_fk`


## 1/ Defining variables 

Databricks SQL let you define your own variable, making it easy to parametrize your sql script.

Variable typically include schema name, or any other string to execute SQL. Here is a basic example:

In [0]:
DECLARE OR REPLACE VARIABLE myvar INT DEFAULT 17;
SELECT myvar;

myvar
17


## 2/ Let's execute multiple statements 

You can now use multi-statements in a script, to delete a record from fact_sales and insert a new record, all in one go:

In [0]:
-- Add and delete records from fact tables multi-statement example 

BEGIN
  DELETE FROM fact_sales WHERE sales_id = 1;
  INSERT INTO fact_sales (product_id, store_id, customer_id, price_sold, units_sold, dollar_cost) VALUES (1, 1, 0, 100.99, 2, 2.99);
  SELECT * FROM fact_sales;
END;

sales_id,product_id,store_id,customer_id,price_sold,units_sold,dollar_cost
2,2,1,0,10.99,2,2.99
3,1,1,0,100.99,2,2.99
4,1,1,10,100.99,2,2.99
5,2,1,10,10.99,2,2.99
6,1,1,0,100.99,2,2.99


## 2/ Stored procedures with DBSQL

In this example, we will create a stored procedure to calculate our revenue. This is a basic example, much more complete procedure can be created:

*Disclaimer: Stored procedures might not be available yet in your workspace - ask your account team - make sure you use the latest DBR version*

In [0]:
-- Stored procedures run only on specific DBR versions and may need to be enabled  
CREATE OR REPLACE PROCEDURE total_revenue(IN price_sold DOUBLE, IN units_sold INT, OUT revenue DOUBLE)
LANGUAGE SQL
SQL SECURITY INVOKER
AS BEGIN
  SET revenue = price_sold * units_sold;
END;

In [0]:
DECLARE revenue DOUBLE DEFAULT 0;
CALL total_revenue(100.99, 3, revenue);
SELECT revenue;

## 3/ Recursive loop example 

In [0]:
WITH RECURSIVE revene_facts AS (
  SELECT 0 AS price_sold, 0 as units_sold , 0 AS revenue
  UNION ALL 
  SELECT  r.price_sold,
          r.units_sold,
          r.price_sold * r.units_sold as revenue
  FROM fact_sales AS r
)

SELECT * FROM revene_facts

## 4/ For loop example 

In [0]:
-- For loop statement 
-- sum all units_sold from a given table
BEGIN
  DECLARE total_units_sold INT DEFAULT 0;
  sumNumbers: FOR row AS SELECT units_sold FROM fact_sales DO
    SET total_units_sold = total_units_sold + row.units_sold;
  END FOR sumNumbers;
  VALUES (total_units_sold);
END;

## 5/ While loop example 

In [0]:

-- sum up all odd numbers from 1 through 10
  BEGIN
    DECLARE sum INT DEFAULT 0;
    DECLARE num INT DEFAULT 0;
    sumNumbers: WHILE num < 10 DO
      SET num = num + 1;
      IF num % 2 = 0 THEN
        ITERATE sumNumbers;
      END IF;
      SET sum = sum + num;
    END WHILE sumNumbers;
    VALUES (sum);
  END;


That's it, you're now ready to migrate your existing SQL script to Databricks!